In [36]:
# Chapter 11
## SNPBLUP
## GBLUP
## Computing SNP solutions from GBLUP
## Computing base population allele frequencies

In [37]:
using DataFrames
using StatsBase
using LinearAlgebra

In [38]:
# Example 11.3 ------------------------------------------------------------

In [39]:
# Animals 21 to 26 are assumed as the selection candidates
a = collect(13:26)
s = [missing, missing, 13, 15, 15, 14, 14, 14, 1, 14, 14, 14, 14, 14]
d = [missing, missing, 4, 2, 5, 6, 9, 9, 3, 8, 11, 10, 7, 12]
dyd = [9.0, 13.4, 12.7, 15.4, 5.9, 7.7, 10.2, 4.8, missing, missing, missing, missing, missing, missing]  

14-element Vector{Union{Missing, Float64}}:
  9.0
 13.4
 12.7
 15.4
  5.9
  7.7
 10.2
  4.8
   missing
   missing
   missing
   missing
   missing
   missing

In [40]:
data = DataFrame(a=a,s=s,d=d,dyd=dyd)

Row,a,s,d,dyd
,Int64,Int64?,Int64?,Float64?
1,13,missing,missing,9.0
2,14,missing,missing,13.4
3,15,13,4,12.7
4,16,15,2,15.4
5,17,15,5,5.9
6,18,14,6,7.7
7,19,14,9,10.2
8,20,14,9,4.8
9,21,1,3,missing


In [41]:
geno = [
    2 0 1 1 0 0 0 2 1 2;
    1 0 0 0 0 2 0 2 1 0;
    1 1 2 1 1 0 0 2 1 2;
    0 0 2 1 0 1 0 2 2 1;
    0 1 1 2 0 0 0 2 1 2;
    1 1 0 1 0 2 0 2 2 1;
    0 0 1 1 0 2 0 2 2 0;
    0 1 1 0 0 1 0 2 2 0;
    2 0 0 0 0 1 2 2 1 2;
    0 0 0 1 1 2 0 2 0 0;
    0 1 1 0 0 1 0 2 2 1;
    1 0 0 0 1 1 0 2 0 0;
    0 0 0 1 1 2 0 2 1 0;
    1 0 1 1 0 2 0 1 0 0
]

14×10 Matrix{Int64}:
 2  0  1  1  0  0  0  2  1  2
 1  0  0  0  0  2  0  2  1  0
 1  1  2  1  1  0  0  2  1  2
 0  0  2  1  0  1  0  2  2  1
 0  1  1  2  0  0  0  2  1  2
 1  1  0  1  0  2  0  2  2  1
 0  0  1  1  0  2  0  2  2  0
 0  1  1  0  0  1  0  2  2  0
 2  0  0  0  0  1  2  2  1  2
 0  0  0  1  1  2  0  2  0  0
 0  1  1  0  0  1  0  2  2  1
 1  0  0  0  1  1  0  2  0  0
 0  0  0  1  1  2  0  2  1  0
 1  0  1  1  0  2  0  1  0  0

In [42]:
geno = Matrix{Union{Float64,Int64}}(geno)

14×10 Matrix{Union{Float64, Int64}}:
 2  0  1  1  0  0  0  2  1  2
 1  0  0  0  0  2  0  2  1  0
 1  1  2  1  1  0  0  2  1  2
 0  0  2  1  0  1  0  2  2  1
 0  1  1  2  0  0  0  2  1  2
 1  1  0  1  0  2  0  2  2  1
 0  0  1  1  0  2  0  2  2  0
 0  1  1  0  0  1  0  2  2  0
 2  0  0  0  0  1  2  2  1  2
 0  0  0  1  1  2  0  2  0  0
 0  1  1  0  0  1  0  2  2  1
 1  0  0  0  1  1  0  2  0  0
 0  0  0  1  1  2  0  2  1  0
 1  0  1  1  0  2  0  1  0  0

In [43]:
p = mean(geno,dims=1) ./ 2

1×10 Matrix{Float64}:
 0.321429  0.178571  0.357143  0.357143  …  0.964286  0.571429  0.392857

In [44]:
#center genotypes
geno .-= 2 .*p

14×10 Matrix{Union{Float64, Int64}}:
  1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
  0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
  0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
  1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
  0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143  -0.357143   0.285714     -0.928571   -1.14286

In [45]:
geno = hcat(a,geno)

14×11 Matrix{Union{Float64, Int64}}:
 13   1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
 14   0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 15   0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 16  -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 17  -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
 18   0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 19  -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 20  -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
 21   1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 22  -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 23  -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
 24   0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 25  -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 26  

In [46]:
# Variance ratios 
varA = 35.241 
varE = 245
k = 2 * (sum(p .* (1.0 .- p)))
αG = varE / varA

6.952129621747397

In [47]:
# Setting up the incidence matrices for the MME

In [48]:
#rows with data
dataRows = .!ismissing.(data.dyd)

14-element BitVector:
 1
 1
 1
 1
 1
 1
 1
 1
 0
 0
 0
 0
 0
 0

In [49]:
X = ones(sum(dataRows))

8-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [50]:
y = data.dyd[dataRows]

8-element Vector{Union{Missing, Float64}}:
  9.0
 13.4
 12.7
 15.4
  5.9
  7.7
 10.2
  4.8

In [51]:
# Prepare genomic relationship matrix G

In [52]:
geno

14×11 Matrix{Union{Float64, Int64}}:
 13   1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
 14   0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 15   0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 16  -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 17  -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
 18   0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 19  -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 20  -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
 21   1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 22  -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 23  -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
 24   0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 25  -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 26  

In [53]:
M = geno[:,2:end]

14×10 Matrix{Union{Float64, Int64}}:
  1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
  0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
  0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
  1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
  0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143  -0.357143   0.285714     -0.928571   -1.14286

In [54]:
G = M*M' ./ k

14×14 Matrix{Float64}:
  1.47224    -0.445566    0.987743  …  -0.142754  -0.829128   -0.263879
 -0.445566    0.745494   -0.930065      0.483057   0.361932    0.361932
  0.987743   -0.930065    1.63374      -0.344629  -0.748378   -0.465753
  0.0591204  -0.445566    0.422495     -0.708003  -0.263879   -0.263879
  0.684932   -0.950252    1.04831      -0.647441  -0.485941   -0.485941
 -0.162942    0.180245   -0.364816  …  -0.364816   0.0793079  -0.203317
 -0.708003    0.200433   -0.627253     -0.344629   0.38212     0.0994953
 -0.546503    0.0793079  -0.183129     -0.183129  -0.0216294  -0.304254
  0.886806    0.0994953   0.119683      0.119683  -0.566691   -0.284066
 -0.788753    0.402307   -0.708003      0.705119   0.866619    0.583994
 -0.203317   -0.142754    0.160058  …  -0.405191  -0.243691   -0.526316
 -0.142754    0.483057   -0.344629      1.06849    0.38212     0.38212
 -0.829128    0.361932   -0.748378      0.38212    0.826244    0.260995
 -0.263879    0.361932   -0.465753      0

In [55]:
inv(G)

14×14 Matrix{Float64}:
  2.15957e15   6.43371e14   9.23068e14  …   1.26162e15   5.14898e14
 -1.17169e15  -1.27548e16  -5.38056e15     -1.06504e16  -1.81884e15
 -6.35273e14  -7.28081e15  -3.321e15       -5.93795e15  -9.91468e14
  5.9155e14    2.11238e15   1.92121e15      1.44755e15   9.17283e14
 -1.24253e15  -6.53558e15  -3.08708e15     -5.6133e15   -8.72324e14
  3.37798e14   2.87907e15   1.62482e15  …   2.23557e15   8.7233e14
 -1.75003e15  -5.00218e15  -3.67987e15     -4.03726e15  -9.62231e14
  2.41332e15  -1.23325e14   1.21946e15      4.73605e14   5.59851e14
  1.54891e14  -2.57349e15  -9.6505e14      -2.01351e15  -1.19141e14
  2.94973e15   5.3507e15    3.27902e15      5.18606e15   1.38722e15
 -9.88606e13  -1.80679e15  -1.26144e15  …  -1.2255e15   -1.64095e14
 -1.92063e15   4.28914e14  -5.59695e14     -2.51555e14   1.93338e14
  2.25736e14  -8.79274e15  -3.25852e15     -7.05061e15  -1.06566e15
  1.54891e14  -2.57349e15  -9.6505e14      -2.01351e15  -1.19141e14

In [56]:
isposdef(G)

false

In [57]:
Ginv = inv(G+Matrix(0.01I,size(G,1),size(G,1)))

14×14 Matrix{Float64}:
  29.0958    -6.45776    2.67571   …  -11.6743      9.4042    7.83732
  -6.45776   51.3134    22.7525        -7.25467    24.5941    4.11045
   2.67571   22.7525    17.1563         0.882643   12.1151    5.75713
  -4.78127    4.5652    -3.04013       23.1712      2.54325   6.80484
  -4.7223    20.741     11.801         14.0265     10.0488    6.67967
   1.2205    -6.89654   -1.36482   …   25.09       -5.60474   7.0263
   7.46202   -0.335171  15.0553        17.4821     -7.46337   5.60085
  20.191      5.46363    4.27085      -13.4382     13.1271    6.69574
   6.55355    6.69023    7.28787        7.07196     7.06325   7.18089
  28.089    -13.1477    -0.454483      -8.17629    -8.59426   4.34234
  15.1065    -6.13869    5.10508   …    8.90612     4.66952   9.72317
 -11.6743    -7.25467    0.882643      47.5223    -11.0858    7.47648
   9.4042    24.5941    12.1151       -11.0858     38.3925   10.7903
   7.83732    4.11045    5.75713        7.47648    10.7903    9.97451

In [58]:
Z = diagm(dataRows)

14×14 BitMatrix:
 1  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  1  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  1  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  1  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  1  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  1  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  1  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  1  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0

In [59]:
Z = Z[dataRows,:]

8×14 BitMatrix:
 1  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  1  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  1  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  1  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  1  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  1  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  1  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  1  0  0  0  0  0  0

In [60]:
# Components of GBLUP MME

In [61]:
X'X

8.0

In [62]:
X'Z

1×14 adjoint(::Vector{Float64}) with eltype Float64:
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0

In [63]:
Z'X

14-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [64]:
Z'Z+Ginv*αG

14×14 Matrix{Float64}:
 203.278   -44.8952    18.6019    -33.24    …  -81.1615    65.3792  54.486
 -44.8952  357.737    158.178      31.7379     -50.4354   170.981   28.5764
  18.6019  158.178    120.273     -21.1354       6.13625   84.2256  40.0243
 -33.24     31.7379   -21.1354    320.658      161.089     17.681   47.3081
 -32.8301  144.194     82.0419     54.3624      97.5141    69.8606  46.4379
   8.4851  -47.9457    -9.48839   192.619   …  174.429    -38.9649  48.8478
  51.8769   -2.33015  104.666    -177.596      121.538    -51.8863  38.9378
 140.371    37.9838    29.6915     72.934      -93.4238    91.2612  46.5496
  45.5611   46.5113    50.6662     48.8782      49.1652    49.1046  49.9225
 195.278   -91.4043    -3.15962    40.0143     -56.8426   -59.7484  30.1885
 105.022   -42.677     35.4911    -59.0979  …   61.9165    32.4631  67.5967
 -81.1615  -50.4354     6.13625   161.089      330.381    -77.0698  51.9775
  65.3792  170.981     84.2256     17.681      -77.0698   266.91  

In [65]:
# MME

In [66]:
LHS = [X'X X'Z;
       Z'X Z'Z+Ginv*αG]

15×15 Matrix{Float64}:
 8.0    1.0       1.0        1.0      …    0.0        0.0      0.0
 1.0  203.278   -44.8952    18.6019      -81.1615    65.3792  54.486
 1.0  -44.8952  357.737    158.178       -50.4354   170.981   28.5764
 1.0   18.6019  158.178    120.273         6.13625   84.2256  40.0243
 1.0  -33.24     31.7379   -21.1354      161.089     17.681   47.3081
 1.0  -32.8301  144.194     82.0419   …   97.5141    69.8606  46.4379
 1.0    8.4851  -47.9457    -9.48839     174.429    -38.9649  48.8478
 1.0   51.8769   -2.33015  104.666       121.538    -51.8863  38.9378
 1.0  140.371    37.9838    29.6915      -93.4238    91.2612  46.5496
 0.0   45.5611   46.5113    50.6662       49.1652    49.1046  49.9225
 0.0  195.278   -91.4043    -3.15962  …  -56.8426   -59.7484  30.1885
 0.0  105.022   -42.677     35.4911       61.9165    32.4631  67.5967
 0.0  -81.1615  -50.4354     6.13625     330.381    -77.0698  51.9775
 0.0   65.3792  170.981     84.2256      -77.0698   266.91    75.0157
 

In [67]:
RHS = [X'y;
       Z'y]

15-element Vector{Union{Missing, Float64}}:
 79.1
  9.0
 13.4
 12.7
 15.4
  5.9
  7.7
 10.2
  4.8
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0

In [68]:
solG = LHS\RHS

15-element Vector{Float64}:
  9.943868574321554
  0.06870096027084555
  0.11549916345806395
  0.048925056861267135
  0.26001296241791944
 -0.4993265718883249
 -0.3589849953562275
  0.1452733959561664
 -0.2310485662921599
  0.02711779520491216
  0.11426180203555535
 -0.23977865264582102
  0.14246103976390093
  0.05365939278384739
  0.35322721743005575

In [69]:
# Backsolve for SNP effects

In [70]:
solSNPfromG = (1.0/k)*M'*Ginv*solG[2:end]

10-element Vector{Union{Float64, Int64}}:
  0.08690846206539095
 -0.31038663841216424
  0.26211279718605135
 -0.08018195974932292
  0.1100558438569419
  0.13889118408636789
 -6.028983288735888e-16
  5.042829888347866e-17
 -0.060602409251707796
 -0.015796726757863706